# 数据索引与选择：



在[第2部分](02.00-Introduction-to-NumPy.ipynb)中，我们详细探讨了访问、设置和修改NumPy数组中值的方法和工具。

这些方法包括索引（例如，`arr[2, 1]`）、切片（例如，`arr[:, 1:5]`）、掩码（例如，`arr[arr > 0]`）、花式索引（例如，`arr[0, [1, 5]]`）及其组合（例如，`arr[:, [1, 5]]`）。

在这里，我们将讨论类似的方式来访问和修改Pandas `Series` 和 `DataFrame` 对象中的值。

如果您使用过NumPy的模式，那么Pandas中的相应模式会让您感到非常熟悉，但也有一些需要注意的小细节。

我们将从一维的简单 `Series` 对象开始，然后再转向更复杂的二维 `DataFrame` 对象。

## 序列中的数据选择

正如您在上一章中所看到的，`Series`对象在许多方面类似于一维NumPy数组，同时也在许多方面类似于标准Python字典。

如果您牢记这两种重叠的类比，将有助于您理解这些数组中的数据索引和选择模式。

### Series as Dictionary

与字典类似，`Series`对象提供了从一组键到一组值的映射。

In [1]:
import pandas as pd
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', 'b', 'c', 'd'])
data

a    0.25
b    0.50
c    0.75
d    1.00
dtype: float64

In [2]:
data['b']

np.float64(0.5)

我们还可以使用类似字典的Python表达式和方法来检查键/索引和值：

In [3]:
'a' in data

True

In [4]:
data.keys()

Index(['a', 'b', 'c', 'd'], dtype='str')

In [5]:
list(data.items())

[('a', 0.25), ('b', 0.5), ('c', 0.75), ('d', 1.0)]

`Series`对象也可以使用类似字典的语法进行修改。

正如您可以通过为新键赋值来扩展字典，您也可以通过为新索引值赋值来扩展`Series`。

In [6]:
data['e'] = 1.25
data

a    0.25
b    0.50
c    0.75
d    1.00
e    1.25
dtype: float64

这种对象的易变性是一个方便的特性：在后台，Pandas正在做出关于内存布局和可能需要进行的数据复制的决策，而用户通常不需要担心这些问题。

### Series 视为一维数组

`Series` 构建在这种类似字典的接口之上，并通过与 NumPy 数组相同的基本机制提供数组风格的项目选择——即切片、掩码和花式索引。



In [7]:
# slicing by explicit index
data['a':'c']

a    0.25
b    0.50
c    0.75
dtype: float64

In [8]:
# slicing by implicit integer index
data[0:2]

a    0.25
b    0.50
dtype: float64

In [9]:
# masking
data[(data > 0.3) & (data < 0.8)]

b    0.50
c    0.75
dtype: float64

In [10]:
# fancy indexing
data[['a', 'e']]

a    0.25
e    1.25
dtype: float64

在这些中，切片可能是最令人困惑的来源。

请注意，当使用显式索引进行切片时（例如，`data['a':'c']`），最后一个索引*包含*在切片中；而当使用隐式索引进行切片时（例如，`data[0:2]`），最后一个索引则*不包含*在切片中。

### 索引器：loc 和 iloc

如果您的 `Series` 具有显式整数索引，则像 `data[1]` 这样的索引操作将使用显式索引，而像 `data[1:3]` 这样的切片操作将使用隐式的 Python 风格索引。

In [11]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data

1    a
3    b
5    c
dtype: str

In [12]:
# explicit index when indexing
data[1]

'a'

In [13]:
# implicit index when slicing
data[1:3]

3    b
5    c
dtype: str

由于整数索引可能导致混淆，Pandas 提供了一些特殊的 *indexer* 属性，以明确暴露某些索引方案。

这些不是功能性方法，而是属性，它们向 `Series` 中的数据提供特定的切片接口。

首先，`loc` 属性允许进行始终引用显式索引的索引和切片。

In [14]:
data.loc[1]

'a'

In [15]:
data.loc[1:3]

1    a
3    b
dtype: str

`iloc` 属性允许进行索引和切片操作，这些操作始终引用隐式的 Python 风格索引。


In [16]:
data.iloc[1]

'b'

In [17]:
data.iloc[1:3]

3    b
5    c
dtype: str

Python代码的一个指导原则是“显式优于隐式”。

`loc`和`iloc`的显式特性使它们在保持代码清晰可读方面非常有帮助；尤其是在整数索引的情况下，一致使用它们可以防止由于混合索引/切片约定而导致的细微错误。

## 数据框中的数据选择

请记住，`DataFrame`在许多方面类似于二维或结构化数组，在其他方面则像是共享相同索引的`Series`结构字典。

在我们探索该结构中的数据选择时，这些类比可能会很有帮助。

### DataFrame作为字典

我们将考虑的第一个类比是将`DataFrame`视为相关`Series`对象的字典。

让我们回到关于各州面积和人口的例子：

In [18]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'Florida': 170312, 'New York': 141297,
                  'Pennsylvania': 119280})
pop = pd.Series({'California': 39538223, 'Texas': 29145505,
                 'Florida': 21538187, 'New York': 20201249,
                 'Pennsylvania': 13002700})
data = pd.DataFrame({'area':area, 'pop':pop})
data

,area,pop
California,423967,39538223
Texas,695662,29145505
Florida,170312,21538187
New York,141297,20201249
Pennsylvania,119280,13002700


个别的 `Series` 组成了 `DataFrame` 的列，可以通过字典风格的索引访问列名：


In [19]:
data['area']

California      423967
Texas           695662
Florida         170312
New York        141297
Pennsylvania    119280
Name: area, dtype: int64

等效地，我们可以使用属性风格的访问方式，列名为字符串：


In [20]:
data.area

California      423967
Texas           695662
Florida         170312
New York        141297
Pennsylvania    119280
Name: area, dtype: int64

尽管这是一种有用的简写，但请记住，它并不适用于所有情况。

例如，如果列名不是字符串，或者列名与`DataFrame`的方法冲突，则无法使用这种属性访问方式。

例如，`DataFrame`具有一个`pop`方法，因此`data.pop`将指向该方法，而不是指向名为 `pop` 的列。

In [21]:
data.pop is data["pop"]

False

特别是，您应该避免通过属性进行列赋值的诱惑（即使用 `data['pop'] = z` 而不是 `data.pop = z`）。

与之前讨论的 `Series` 对象一样，这种字典风格的语法也可以用来修改对象，在这种情况下添加新列。

In [22]:
data['density'] = data['pop'] / data['area']
data

,area,pop,density
California,423967,39538223,93.257784
Texas,695662,29145505,41.896072
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120
Pennsylvania,119280,13002700,109.009893


这段文字展示了 `Series` 对象之间逐元素算术运算的简单语法预览；我们将在[在Pandas中操作数据](03.03-Operations-in-Pandas.ipynb)中进一步探讨这一内容。

### DataFrame作为二维数组

如前所述，我们也可以将`DataFrame`视为一种增强的二维数组。

我们可以使用`values`属性来检查原始底层数据数组。

In [23]:
data.values

array([[4.23967000e+05, 3.95382230e+07, 9.32577842e+01],
       [6.95662000e+05, 2.91455050e+07, 4.18960717e+01],
       [1.70312000e+05, 2.15381870e+07, 1.26463121e+02],
       [1.41297000e+05, 2.02012490e+07, 1.42970120e+02],
       [1.19280000e+05, 1.30027000e+07, 1.09009893e+02]])

考虑到这一点，我们可以在 `DataFrame` 上执行许多熟悉的数组操作。

例如，我们可以对整个 `DataFrame` 进行转置，以交换行和列。

In [24]:
data.T

,California,Texas,Florida,New York,Pennsylvania
area,4.239670e+05,6.956620e+05,1.703120e+05,1.412970e+05,1.192800e+05
pop,3.953822e+07,2.914550e+07,2.153819e+07,2.020125e+07,1.300270e+07
density,9.325778e+01,4.189607e+01,1.264631e+02,1.429701e+02,1.090099e+02


当涉及到`DataFrame`对象的索引时，很明显，字典式列索引阻碍了我们将其简单地视为NumPy数组的能力。

特别是，向数组传递单个索引会访问一行：

In [25]:
data.values[0]

array([4.23967000e+05, 3.95382230e+07, 9.32577842e+01])

并且将单个“索引”传递给 `DataFrame` 可以访问一列：

In [26]:
data['area']

California      423967
Texas           695662
Florida         170312
New York        141297
Pennsylvania    119280
Name: area, dtype: int64

因此，对于数组样式的索引，我们需要另一种约定。

在这里，Pandas再次使用之前提到的 `loc` 和 `iloc` 索引器。

通过使用 `iloc` 索引器，我们可以像处理简单的 NumPy 数组一样对底层数组进行索引（采用隐式的 Python 风格索引），但结果中保留了 `DataFrame` 的行索引和列标签。

In [27]:
data.iloc[:3, :2]

,area,pop
California,423967,39538223
Texas,695662,29145505
Florida,170312,21538187



同样，使用 `loc` 索引器，我们可以以数组类似的方式对底层数据进行索引，但需明确指定索引和列名。

In [28]:
data.loc[:'Florida', :'pop']

,area,pop
California,423967,39538223
Texas,695662,29145505
Florida,170312,21538187


任何熟悉的 NumPy 风格数据访问模式都可以在这些索引器中使用。

例如，在 `loc` 索引器中，我们可以如下结合掩码和花式索引：

In [29]:
data.loc[data.density > 120, ['pop', 'density']]

,pop,density
Florida,21538187,126.463121
New York,20201249,142.970120


任何这些索引约定也可以用于设置或修改值；这以您可能习惯于使用NumPy的标准方式进行。

In [30]:
data.iloc[0, 2] = 90
data

,area,pop,density
California,423967,39538223,90.000000
Texas,695662,29145505,41.896072
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120
Pennsylvania,119280,13002700,109.009893


为了提高您在Pandas数据处理方面的流利度，我建议您花一些时间与一个简单的`DataFrame`进行互动，探索这些不同索引方法所允许的索引、切片、掩码和花式索引类型。

### 额外的索引约定

有几个额外的索引约定可能与前面的讨论相悖，但在实践中仍然可以非常有用。

首先，*索引*指的是列，而*切片*则指的是行。

In [31]:
data['Florida':'New York']

,area,pop,density
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120


这样的切片也可以通过数字而不是索引来引用行：

In [32]:
data[1:3]

,area,pop,density
Texas,695662,29145505,41.896072
Florida,170312,21538187,126.463121


同样，直接掩蔽操作是按行而不是按列进行解释的：

In [33]:
data[data.density > 120]

,area,pop,density
Florida,170312,21538187,126.463121
New York,141297,20201249,142.970120


这两种约定在语法上与NumPy数组的约定相似，尽管它们可能并不完全符合Pandas约定的模式，但由于其实际效用，它们被纳入考虑。